# SQL Queries

## 1. Show the total number of flights for each aircraft model, listing the model and its count.

In [2]:
import pandas as pd

In [5]:
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345",
    )
cursor =  connection.cursor()
cursor

In [6]:
query = "use AeroDataBox"
cursor.execute(query)

In [ ]:
query = "SELECT ROW_NUMBER() OVER () AS no, model, COUNT(flight_number) AS Total_flights FROM aircraft JOIN flight ON aircraft_code=aircraft_registration  GROUP BY model"

df=pd.read_sql(query, connection)
print(df.to_string(index=False))
#verified manually using SELECT model, aircraft_code, aircraft_registration, COUNT( aircraft_registration) as total_flight FROM flight JOIN aircraft on aircraft_code=aircraft_registration where model = 'A20N' group by model, aircraft_code, aircraft_registration;

 no model  Total_flights
  1  B744              1
  2  A20N             17
  3  A320             16
  4  A21N             14
  5  AT75              2
  6  A321              6
  7  E170              1
  8  B738              8
  9  B789              7
 10  E175              1
 11  A319              5
 12  A388              7
 13  B773             12
 14  B772              1
 15  B781              1
 16   777              2
 17  A35K              1
 18  A333              2
 19  CRJ9              3
 20  B752              1
 21  A359              1
 22  B38M              5
 23  B788              2
 24  A332              1


C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\3108856341.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


## 2. List all aircraft (registration, model) that have been assigned to more than 5 flights

In [ ]:
#since flight count is not more than 4, i used greater than or equal to 3
query = """SELECT aircraft_registration, model, COUNT(*) AS flight_count 
FROM aircraft JOIN flight 
ON aircraft_code = aircraft_registration 
GROUP BY aircraft_code, model 
HAVING COUNT(*) >=3"""
df=pd.read_sql(query, connection)
df


C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\3830913986.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,aircraft_registration,model,flight_count
0,G-EUYV,A320,4
1,G-XLED,A388,4
2,EI-SIA,A20N,3
3,4X-EDA,B789,3
4,G-EUYC,A320,3
5,D-AGWV,A319,4


## 3. For each airport, display its name and the number of outbound flights, but only for airports with more than 5 flights.

In [ ]:
query = "SELECT icao_code, name, COUNT(flight_number) as number_of_outbound_flights from airport join flight on icao_code=origin_icao GROUP BY icao_code having count(flight_number)>5"
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\4238549280.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,icao_code,name,number_of_outbound_flights
0,YSSY,Sydney Kingsford Smith,20
1,VOBL,Bangalore Bengaluru,21
2,OMDB,Dubai,22
3,VIDP,New Delhi Indira Gandhi,21
4,EHAM,Amsterdam Schiphol,22
5,VTCC,Chiangmai Chiang Mai,20
6,EDDM,Munich,21
7,KJFK,New York John F Kennedy,21
8,EGLL,London Heathrow,22
9,CYVR,Vancouver,20


## 4. Find the top 3 destination airports (name, city) by number of arriving flights, sorted by count descending.

In [ ]:
query = """SELECT destination_icao, name, city, COUNT(flight_number) AS arriving_flights
 FROM flight JOIN airport ON destination_icao = icao_code
   GROUP BY icao_code 
   ORDER BY arriving_flights DESC
     LIMIT 3"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\2126184087.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,destination_icao,name,city,arriving_flights
0,WSSS,Singapore Changi,Singapore,28
1,LFPG,Paris Charles de Gaulle,Paris,26
2,EGLL,London Heathrow,London,24


## 5. Show for each flight: number, origin, destination, and a label 'Domestic' or 'International' using CASE WHEN on country match

In [ ]:
query = """SELECT flight_number, origin_icao, destination_icao,
CASE 
WHEN airport1.country = airport2.country 
THEN 'Domestic' ELSE 'International'
END AS flight_type
 FROM flight 
 JOIN airport AS airport1 on origin_icao=airport1.icao_code 
  JOIN airport AS airport2 on destination_icao = airport2.icao_code """
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\1675702155.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,flight_number,origin_icao,destination_icao,flight_type
0,6E 1657,VIDP,OMDB,International
1,6E 3223,EHAM,LFPG,International
2,6E 817,VOBL,VIDP,Domestic
3,6E 861,VIDP,VOBL,Domestic
4,AA 141,EGLL,KJFK,International
5,AA 292,KJFK,VIDP,International
6,AC 5944,WSSS,WMKP,International
7,AC 5948,WMKP,WSSS,International
8,AC 7610,OMDB,WSSS,International
9,AC 860,CYVR,EGLL,International


## 6. Show the 5 most recent arrivals at “DEL” airport including flight number, aircraft, departure airport name, and arrival time, ordered by latest arrival.

In [ ]:
query = """SELECT city, flight_number, aircraft_code AS aircraft, name AS departure_airport_name, 
actual_arrival, DATE_FORMAT(STR_TO_DATE(actual_arrival, '%Y-%m-%d %H:%iZ'), '%H:%i') AS arrival_time 
FROM airport 
JOIN flight ON airport.icao_code = flight.origin_icao
JOIN aircraft ON aircraft.aircraft_code=flight.aircraft_registration 
WHERE iata_code = 'DEL'
ORDER BY arrival_time DESC
LIMIT 5"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\1803676126.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,city,flight_number,aircraft,departure_airport_name,actual_arrival,arrival_time
0,New Delhi,AI 358,VT-TSN,New Delhi Indira Gandhi,2026-05-17 22:08Z,22:08
1,New Delhi,MU 564,B-8226,New Delhi Indira Gandhi,2026-05-17 20:11Z,20:11
2,New Delhi,6E 1701,VT-IPG,New Delhi Indira Gandhi,2026-05-17 20:00Z,20:00
3,New Delhi,6E 1657,VT-IPY,New Delhi Indira Gandhi,2026-05-17 18:12Z,18:12
4,New Delhi,6E 772,VT-IWN,New Delhi Indira Gandhi,2026-05-17 18:00Z,18:00


## 7. Find all airports with no arriving flights (never used as a destination in flights table)

In [ ]:
query = """SELECT icao_code, name, city
FROM airport
LEFT JOIN flight ON airport.icao_code = flight.destination_icao
WHERE flight.destination_icao= 'None'"""
df= pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_13256\2695690141.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df= pd.read_sql(query, connection)


,icao_code,name,city


## 8. For each airline, count the number of flights by status (e.g., 'On Time', 'Delayed', 'Cancelled') using CASE WHEN.

In [ ]:
query = """WITH base AS ( SELECT
        aircraft.airline,
        airport_delays.dep_status,
        airport_delays.arr_status

    FROM airport_delays

    JOIN flight
    ON airport_delays.flight_number = flight.flight_number

    JOIN aircraft
ON aircraft.aircraft_code = flight.aircraft_registration)

SELECT
    airline,

    SUM(
        CASE
            WHEN dep_status = 'delayed'
            THEN 1
            ELSE 0
        END
    ) AS delayed_departure,

    SUM(
        CASE
            WHEN dep_status = 'on_time'
            THEN 1
            ELSE 0
        END
    ) AS on_time_departure,

    SUM(
        CASE
            WHEN arr_status = 'delayed'
            THEN 1
            ELSE 0
        END
    ) AS delayed_arrival,

    SUM(
        CASE
            WHEN arr_status = 'on_time'
            THEN 1
            ELSE 0
        END
    ) AS on_time_arrival

FROM base

GROUP BY airline"""

df= pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_26360\3829173558.py:54: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df= pd.read_sql(query, connection)


,airline,delayed_departure,on_time_departure,delayed_arrival,on_time_arrival
0,IndiGo,2.0,14.0,3.0,13.0
1,American Airlines,5.0,0.0,0.0,5.0
2,Republic Airways,2.0,0.0,0.0,2.0
3,British Airways,15.0,1.0,9.0,7.0
4,Air India,2.0,3.0,1.0,4.0
5,Vistara,0.0,1.0,0.0,1.0
6,SAS Ireland,3.0,0.0,0.0,3.0
7,El Al,0.0,3.0,0.0,3.0
8,JetBlue Airways,4.0,8.0,1.0,11.0
9,Cathay Pacific,0.0,2.0,0.0,2.0


## 9. Show all canceled flights, with aircraft and both airports, ordered by departure time descending

In [32]:
query = """
SELECT flight_number, origin_icao, destination_icao, scheduled_departure, status
FROM flight
WHERE status = 'canceled'
ORDER BY STR_TO_DATE(
            flight.scheduled_departure,
            '%Y-%m-%d %H:%iZ'
         ) DESC"""

pd.read_sql(query, connection)

C:\Users\hamsa\AppData\Local\Temp\ipykernel_26360\1434106660.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, connection)


,flight_number,origin_icao,destination_icao,scheduled_departure,status
0,VZ 119,VTCC,VTBS,2026-05-17 15:30Z,Canceled
1,FD 8439,VTBD,VTCC,2026-05-17 13:45Z,Canceled
2,VZ 118,VTBS,VTCC,2026-05-17 13:35Z,Canceled
3,JL 2060,RJFF,RJOO,2026-05-17 10:00Z,Canceled


## 10. List all city pairs (origin-destination) that have more than 2 different aircraft models operating flights between them

In [ ]:
query = """SELECT airport_departure.city AS origin_city,
    airport_arrival.city AS destination_city,
    COUNT(DISTINCT aircraft.model) AS aircraft_model_count
FROM flight JOIN airport airport_departure
    ON flight.origin_icao = airport_departure.icao_code
JOIN airport airport_arrival ON flight.destination_icao = airport_arrival.icao_code
JOIN aircraft ON flight.aircraft_registration = aircraft.aircraft_code
GROUP BY airport_departure.city, airport_arrival.city 
HAVING COUNT(DISTINCT aircraft.model) >= 2"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_20248\2828516602.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,origin_city,destination_city,aircraft_model_count
0,Bangalore,New Delhi,2
1,London,New York,2
2,New Delhi,Bangalore,2


## 11. For each destination airport, compute the % of delayed flights (status='Delayed') among all arrivals, sorted by highest percentage

In [7]:
query = """SELECT flight.destination_icao AS destination_airport, COUNT(*) AS total_arrivals, 
    SUM(CASE
            WHEN airport_delays.arr_status = 'delayed' THEN 1 ELSE 0
        END) AS delayed_arrivals,
    ROUND(100.0 *
        SUM(CASE
                WHEN airport_delays.arr_status = 'delayed' THEN 1 ELSE 0
            END) / COUNT(*), 2
    ) AS delayed_percentage
FROM airport_delays JOIN flight ON airport_delays.flight_number = flight.flight_number
GROUP BY flight.destination_icao
ORDER BY delayed_percentage DESC"""
df=pd.read_sql(query, connection)
df


C:\Users\hamsa\AppData\Local\Temp\ipykernel_26360\1926611077.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,destination_airport,total_arrivals,delayed_arrivals,delayed_percentage
0,KLAS,1,1.0,100.0
1,CYQQ,1,1.0,100.0
2,KSFO,2,2.0,100.0
3,LROP,5,5.0,100.0
4,LOWW,2,2.0,100.0
...,...,...,...,...
124,LTFM,1,0.0,0.0
125,YPPH,1,0.0,0.0
126,LRTR,1,0.0,0.0
127,LEBL,1,0.0,0.0
